In [1]:
import sys
import os.path as osp
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root.parent / "hostApps" / "python" ))
from pdcv2_modules.pixMap import xy_map

In [2]:
datadir = Path("~/Desktop/UCSDPDCv2/data").expanduser()
filedir = datadir.glob("TCR*RT")
files = [list(folder.glob("*.csv"))[0] for folder in filedir]

print(f"Running {len(files)} files")

Running 9 files


In [ ]:
pdc_layout = {
    0: (0,0),
    1: (64, 0),
    2: (64, 64),
    3: (0, 64),
}

def read_data(file: Path | str):
    print(f"Running file {file}")
    df = pd.read_csv(file, delimiter=";")
    pdc_num = len(df.columns) // 4
    assert pdc_num == 4, f"Needs 4 PDCs. {pdc_num} found in {file}"
    return df, pdc_num

def format_data(data: pd.DataFrame, pdcs: int):
    grid = np.full((128, 128), np.nan)
    for pdc in range(pdcs):
        x_offset, y_offset = pdc_layout[pdc]
        for idx, val in zip(data[f"SPAD_idx{pdc}"], data[f"SPAD_TCR{pdc}"]):
            xy = xy_map(idx)
            if xy is not None:
                x, y = xy
                grid[y+y_offset, x+x_offset] = float(val)

    return grid
        
def draw_heatmaps(files: list[Path], save: str|Path|bool = False, show: bool = True):
    for file in files:
        data, pdcs = read_data(file)
        grid = format_data(data, pdcs)

        fig, ax = plt.subplots()
        im = ax.imshow(grid, origin="lower", norm=LogNorm())
        plt.colorbar(im, label="TCR")

        ax.axvline(64-0.5, color="black", linewidth=1)
        ax.axhline(64-0.5, color="black", linewidth=1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title("Pixel TCR")

        if save:
            savepath = file.with_suffix(".png")
            plt.savefig(savepath)
            print(f"File saved to {savepath}")
        if show:
            plt.show()
        plt.close()

draw_heatmaps(files, True, False)

Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_21.5_RT\20261016_12h07m03_TCR_H62_200ms_21V5_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_21.5_RT\20261016_12h07m03_TCR_H62_200ms_21V5_PDC0_PDC1_PDC2_PDC3.png
Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_22.5_RT\20261016_12h53m57_TCR_H62_200ms_22V5_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_22.5_RT\20261016_12h53m57_TCR_H62_200ms_22V5_PDC0_PDC1_PDC2_PDC3.png
Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_22_RT\20261016_12h30m32_TCR_H62_200ms_22V_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_22_RT\20261016_12h30m32_TCR_H62_200ms_22V_PDC0_PDC1_PDC2_PDC3.png
Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_23.5_RT\20261016_09h53m22_TCR_H62_200ms_23V5_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\TCR_101626_23.5_RT\20261016_09h53m22_TCR_H62_200ms